In [1]:
# 1) ติดตั้ง (ครั้งแรกใน Colab เท่านั้น)
!pip install -q pandas scikit-learn joblib openpyxl

In [2]:
# 2) โหลดไฟล์ (หรือ DataFrame ที่สร้างเอง)
import pandas as pd
df = pd.read_excel('/content/sample_data/data.xlsx', sheet_name='Sheet8')

print(df)

        order_id shop_guid  app_source                  order_time  \
0      ORD900000  SHOP_007  ShopeeFood  2025-04-28 01:04:49.000000   
1      ORD900001  SHOP_056    LINE_MAN  2025-04-28 07:08:04.000000   
2      ORD900002  SHOP_059    LINE_MAN  2025-04-27 18:17:48.000000   
3      ORD900003  SHOP_018    GrabFood  2025-04-27 16:56:23.000000   
4      ORD900004  SHOP_055    GrabFood  2025-04-27 19:40:53.000000   
...          ...       ...         ...                         ...   
99995  ORD999995  SHOP_023  ShopeeFood  2025-04-28 01:01:19.000000   
99996  ORD999996  SHOP_023    GrabFood  2025-04-28 06:56:17.000000   
99997  ORD999997  SHOP_006   Robinhood  2025-04-27 21:29:21.000000   
99998  ORD999998  SHOP_011    LINE_MAN  2025-04-28 04:32:13.000000   
99999  ORD999999  SHOP_059  ShopeeFood  2025-04-27 17:44:18.000000   

       distance_km  queue_size  prep_ahead_sum_min  prep_time_min  \
0             2.72           3                   8              7   
1             6.80   

In [3]:
# 3) เตรียมฟีเจอร์
FEATURES = ['distance_km','queue_size','prep_ahead_sum_min',
            'prep_time_min','traffic_sec','rain_flag']
df[FEATURES] = df[FEATURES].fillna(0)      # กัน NaN
X, y = df[FEATURES], df['eta_actual_min']
print(X)
print(y)


       distance_km  queue_size  prep_ahead_sum_min  prep_time_min  \
0             2.72           3                   8              7   
1             6.80           4                  11              4   
2             0.89           5                  17              8   
3             1.17           3                   6              3   
4             7.45           1                   4              4   
...            ...         ...                 ...            ...   
99995         3.08           3                   8              3   
99996         2.15           3                   9              3   
99997         4.01           3                  11              3   
99998         0.58           5                  19              4   
99999         2.64           2                   7              7   

       traffic_sec  rain_flag  
0              349          0  
1              874          0  
2              114          0  
3              150          0  
4          

In [4]:
# 4) split
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
# 5) Pipeline (สเกล + Boosting)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
pipe = Pipeline([
    ('sc', StandardScaler()),
    ('gb', GradientBoostingRegressor(n_estimators=400,
                                     learning_rate=0.05,
                                     max_depth=3,
                                     random_state=42))
]).fit(X_tr, y_tr)

In [6]:
# 6) ประเมิน
from sklearn.metrics import mean_absolute_error, mean_squared_error
import math

pred = pipe.predict(X_te)
mae = mean_absolute_error(y_te, pred)  # Calculate MAE
print('MAE :', round(mae, 2), 'นาที') # Apply round to the MAE value
# Calculate RMSE using math.sqrt if squared parameter is not available
rmse = math.sqrt(mean_squared_error(y_te, pred))
print('RMSE:', round(rmse, 2), 'นาที')

MAE : 1.21 นาที
RMSE: 1.51 นาที


In [7]:
# 7) บันทึกโมเดล
import joblib # Import the joblib library
joblib.dump(pipe, 'eta_model.pkl')

['eta_model.pkl']

In [9]:
# 8) ทดสอบออเดอร์ใหม่
new_order = pd.DataFrame([{
    'distance_km':3.2,
    'queue_size':4,
    'prep_ahead_sum_min':20,
    'prep_time_min':13,
    'traffic_sec':500,
    'rain_flag':0
}])[FEATURES]            # เรียงคอลัมน์ให้ตรง
eta_pred = pipe.predict(new_order)[0]
print(f"ETA คาด ≈ {eta_pred:.1f} นาที")

ETA คาด ≈ 34.7 นาที


# ส่วนใหม่